**Import Libraries and Setup**

In [ ]:
# ============================================
# MLP / ANN - Credit Card Fraud Detection
# SE4050 - Deep Learning
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

TensorFlow version: 2.20.0
NumPy version: 2.1.3
Pandas version: 2.2.3


**Set the Random Seed as 42**

In [ ]:
# Reproducibility
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed set to:", SEED)

Random seed set to: 42


**Upload your cleaned CSV**

In [ ]:
import os

print(os.listdir('/content'))

['.config', 'sample_data']


It seems the `creditcard_cleaned.csv` file is located inside the `sample_data` directory. Let's list the contents of that directory to confirm.

In [ ]:
print(os.listdir('/content/sample_data'))

['anscombe.json', 'README.md', 'mnist_train_small.csv', 'california_housing_test.csv', 'mnist_test.csv', 'california_housing_train.csv']


In [ ]:
file_path = "/content/sample_data/creditcard_cleaned.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (27825, 31)


**Check the target distribution**

In [ ]:
class_counts = df["Class"].value_counts()
print(class_counts)

print()

class_percentages = df["Class"].value_counts(normalize=True) * 100
print(class_percentages)

# print()

# plt.figure(figsize=(6, 4))

# df["Class"].value_counts().plot(kind="bar")

# plt.title("Class Distribution")
# plt.xlabel("Class")
# plt.ylabel("Number of Transactions")
# plt.xticks(rotation=0)

# plt.show()

Class
0.0    27731
1.0       93
Name: count, dtype: int64

Class
0.0    99.665756
1.0     0.334244
Name: proportion, dtype: float64


**Separate X and y (30 input features → 1 binary output.)**


In [ ]:
# Separate input features and target

X = df.drop("Class", axis=1)
y = df["Class"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (27825, 30)
y shape: (27825,)


**Create Train / Validation / Test sets**


*   Training (70% Training)  → train the MLP

*   Validation(15% Validation) → make decisions/tune the model
*   Test(15% Test)       → final evaluation ONLY



In [ ]:
import pandas as pd

# Drop rows where 'Class' (target variable) is NaN
# This is necessary because train_test_split with stratify=y does not support NaN values in y.
# Ensure X and y are aligned after dropping NaNs.

# From the kernel state, X and y have 27825 rows, and the last one is NaN.
# df also has 27825 rows, and the last one is NaN across all columns.

# Re-create X and y from the original df after dropping NaNs
clean_df = df.dropna(subset=['Class'])
X_cleaned = clean_df.drop("Class", axis=1)
y_cleaned = clean_df["Class"]


# First split: 85% temporary training data, 15% test data
X_temp, X_test, y_temp, y_test = train_test_split(
    X_cleaned, # Use the cleaned X
    y_cleaned, # Use the cleaned y
    test_size=0.15,
    random_state=SEED,
    stratify=y_cleaned # Stratify with the cleaned y
)

# Second split: split the 85% temporary data into
# approximately 70% training and 15% validation

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.17647,
    random_state=SEED,
    stratify=y_temp
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

Training set: (19476, 30) (19476,)
Validation set: (4174, 30) (4174,)
Test set: (4174, 30) (4174,)


**Check fraud counts in each split**

In [ ]:
print("Training class distribution:")
print(y_train.value_counts())
print()

print("Validation class distribution:")
print(y_val.value_counts())
print()

print("Test class distribution:")
print(y_test.value_counts())

Training class distribution:
Class
0.0    19411
1.0       65
Name: count, dtype: int64

Validation class distribution:
Class
0.0    4160
1.0      14
Name: count, dtype: int64

Test class distribution:
Class
0.0    4160
1.0      14
Name: count, dtype: int64


**Standardize the features**

MLPs generally train better when numerical features are on comparable scales.

In [ ]:
# Create scaler
scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Use the same fitted scaler for validation and test
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Training scaled shape:", X_train_scaled.shape)
print("Validation scaled shape:", X_val_scaled.shape)
print("Test scaled shape:", X_test_scaled.shape)

Training scaled shape: (19476, 30)
Validation scaled shape: (4174, 30)
Test scaled shape: (4174, 30)


**Calculate class weights**

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# Calculate class weights using ONLY the training labels

classes = np.unique(y_train)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, class_weights_array))

print("Class weights:")
print(class_weights)

Class weights:
{np.float64(0.0): np.float64(0.5016743083818453), np.float64(1.0): np.float64(149.8153846153846)}


**MLP/ANN architecture**

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential([
    Dense(64, activation="relu", input_shape=(30,)),
    Dropout(0.3),

    Dense(32, activation="relu"),
    Dropout(0.3),

    Dense(16, activation="relu"),

    Dense(1, activation="sigmoid")
])

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,609 (18.00 KB)

 Trainable params: 4,609 (18.00 KB)

 Non-trainable params: 0 (0.00 B)

**MLP Model Compilation**

Optimizer, loss function, and evaluation metrics define(Configures how the MLP model should learn during training.)



*   Adam → Determines how the model’s weights should be updated during training.

*   Binary Cross-Entropy → Measures how incorrect the model’s predictions are.
*   Accuracy → Measures how many predictions are correct.


*   Precision → Measures how many of the transactions predicted as fraud are actually fraud.

*   Recall → Measures how many of the actual fraud transactions are correctly detected.
*   AUC → Measures how well the model distinguishes between normal and fraudulent transactions.

Simple Goal:

This code prepares the MLP model for training by defining how the model should learn and how its performance should be measured.

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

print("MLP model compiled successfully!")

MLP model compiled successfully!


**Train the MLP/ANN**

Now we train the model using your training set, validate it using the validation set, and apply our calculated class_weights.

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=20,
    batch_size=256,
    class_weight=class_weights,
    verbose=1
)

Epoch 1/20
77/77 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.7516 - auc: 0.8906 - loss: 0.4287 - precision: 0.0113 - recall: 0.8462 - val_accuracy: 0.9974 - val_auc: 0.9955 - val_loss: 0.3515 - val_precision: 0.5789 - val_recall: 0.7857
Epoch 2/20
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9865 - auc: 0.9660 - loss: 0.2171 - precision: 0.1867 - recall: 0.9077 - val_accuracy: 0.9976 - val_auc: 0.9994 - val_loss: 0.1300 - val_precision: 0.5833 - val_recall: 1.0000
Epoch 3/20
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9898 - auc: 0.9877 - loss: 0.1325 - precision: 0.2362 - recall: 0.9231 - val_accuracy: 0.9976 - val_auc: 0.9995 - val_loss: 0.0778 - val_precision: 0.5833 - val_recall: 1.0000
Epoch 4/20
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9920 - auc: 0.9976 - loss: 0.0809 - precision: 0.2936 - recall: 0.9846 - val_accuracy: 0.9988 - val_auc: 0.9995 - val_loss: 0.0439 - val_precision: 0.7368 - val_recall: 1.0000
Epoch 5/20
77/77 ━━━━━━━━━━━━━━━━━━━━ 0

**What your results show**
| Metric    | Training | Validation |
| --------- | -------: | ---------: |
| Accuracy  |   98.16% | **98.49%** |
| AUC       |   99.79% | **96.97%** |
| Precision |    8.09% |  **9.04%** |
| Recall    |   96.98% | **88.73%** |
| Loss      |   0.0598 | **0.0398** |

